In [ ]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
import boto3
import pickle
from api import map_ltv_range_to_lgd_bin

try:
    import optbinning
except:
    ! pip install optbinning
    
try:
    import catboost
except:
    ! pip install catboost

try:
    import xmltodict
except:
    ! pip install xmltodict

#### Functions

In [ ]:
def get_lgd_bk_nobk(int_bk, flt_lgd_bk, flt_lgd_nobk):
    # if bk
    if int_bk == 1:
        return flt_lgd_bk
    else:
        return flt_lgd_nobk

#### Constants

In [ ]:
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')

str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')

str_dirname_output = './output'

#### Output dir

In [ ]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

#### Import data

In [ ]:
str_filename = 'df.gzip'
str_uri = f's3://20241112-simple-model-test/08_prep_data/{str_filename}'
df = pd.read_parquet(
    str_uri,
)
# sort
df.sort_values(by='request_datetime', ascending=True, inplace=True)
df

#### Import parser

In [ ]:
str_filename = 'cls_parser.pkl'
str_local_path = f'./{str_filename}'
cls_parser = pickle.load(open(str_local_path, 'rb'))

#### Preprocess data

In [ ]:
cls_model_preprocessing = cls_parser.cls_model_preprocessing
df = cls_model_preprocessing.transform(df)
# show
df

#### PD predictions

In [ ]:
cls_model_inference = cls_parser.cls_model_inference
# get intercept
flt_intercept = cls_model_inference.intercept_[0]
# get cols in model
list_cols_model = list(cls_model_inference.feature_names_in_)
# get the coef
list_coef = list(cls_model_inference.coef_[0])
# make dict
dict_coef = dict(zip(list_cols_model, list_coef))
# get contribution
list_str_contribution = []
for key, val in dict_coef.items():
    str_contribution = f'{key}_contribution'
    df[str_contribution] = df[key] * val 
    list_str_contribution.append(str_contribution)
# get the sum
df['sum'] = df[list_str_contribution].apply(sum, axis=1)
# get the log odds
df['log_odds'] = df['sum'] + flt_intercept
# get the pd
df['pd'] = np.exp(df['log_odds']) / (1 + np.exp(df['log_odds']))
# show
df

#### LGD

In [ ]:
# bk
dict_bins_ltv = cls_parser.dict_bins_ltv_bk
df['lgd_bk'] = df['ENG-loan_to_value'].apply(
    lambda x: map_ltv_range_to_lgd_bin(
        flt_ltv=x,
        dict_bins_ltv=dict_bins_ltv,
    ),
)
# show
#df

In [ ]:
# non bk
dict_bins_ltv = cls_parser.dict_bins_ltv_nobk
df['lgd_nobk'] = df['ENG-loan_to_value'].apply(
    lambda x: map_ltv_range_to_lgd_bin(
        flt_ltv=x,
        dict_bins_ltv=dict_bins_ltv,
    ),
)
# show
#df

In [ ]:
# choose appropriate ltv
df['lgd'] = df.apply(
    lambda x: get_lgd_bk_nobk(
        int_bk=x['ENG-bk'],
        flt_lgd_bk=x['lgd_bk'],
        flt_lgd_nobk=x['lgd_nobk'],
    ),
    axis=1,
)
# show
df

#### Convert non-numeric to string

In [ ]:
for col in tqdm(df.columns):
    str_dtype = df[col].dtype
    if str_dtype not in ['int64','float64']:
        df[col] = df[col].astype(str)
    else:
        pass

#### Write to s3

In [ ]:
%%time

str_filename = 'df_clean_w_pred.gzip'
str_uri = f's3://{str_project}/{str_task}/{str_filename}'
df.to_parquet(
    str_uri,
    compression='gzip',
)